In [36]:
import pandas as pd
games = pd.read_csv("../data/games_2025_26.csv")

# 1: Explore
games.shape
games.columns
games.describe()
games.head()
games.dtypes

SEASON_ID              int64
TEAM_ID                int64
TEAM_ABBREVIATION        str
TEAM_NAME                str
GAME_ID                int64
GAME_DATE                str
MATCHUP                  str
WL                       str
MIN                    int64
PTS                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PLUS_MINUS           float64
dtype: object

In [ ]:
# 2: Filter by 140 points or more and sort by points scored
games_140 = games[games['PTS'] >= 140]
games_140 = games_140.sort_values(by='PTS', ascending=False)    
games_140.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
793,22025,1610612743,DEN,Denver Nuggets,22500810,2026-02-20,DEN @ POR,W,240,157,...,0.813,13,47,60,41,12,3,15,22,54.0
301,22025,1610612755,PHI,Philadelphia 76ers,22501053,2026-03-25,PHI vs. CHI,W,239,157,...,0.750,9,36,45,34,9,10,8,24,20.0
93,22025,1610612740,NOP,New Orleans Pelicans,22501153,2026-04-07,NOP vs. UTA,W,240,156,...,0.870,10,38,48,32,15,5,16,18,19.0
178,22025,1610612755,PHI,Philadelphia 76ers,22501105,2026-04-01,PHI @ WAS,W,240,153,...,0.875,10,31,41,37,9,2,10,18,22.0
504,22025,1610612746,LAC,LA Clippers,22500952,2026-03-11,LAC vs. MIN,W,240,153,...,0.833,7,22,29,36,16,4,10,24,25.0


In [22]:
# 3: Group and summarize points per game and win percentage by each team, sorted from best to worst
games['WIN'] = (games['WL'] == 'W').astype(int)
best_teams = games.groupby('TEAM_ABBREVIATION')
best_teams = best_teams.agg(avg_points=('PTS', 'mean'),win_pct=('WIN', 'mean'))
best_teams = best_teams.sort_values('win_pct', ascending=False)
best_teams.head()

,avg_points,win_pct
TEAM_ABBREVIATION,,
OKC,119.024390,0.780488
SAS,119.829268,0.756098
DET,117.768293,0.731707
BOS,114.853659,0.682927
DEN,122.073171,0.658537


In [51]:
# 4 Add is_home column (TRUE/FALSE)
matchups = games.copy()
matchups['is_home'] = matchups['MATCHUP'].str.contains('vs.')
matchups.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,is_home
0,22025,1610612761,TOR,Toronto Raptors,22501192,2026-04-12,TOR vs. BKN,W,239,136,...,3,40,43,36,9,8,10,20,35.0,True
1,22025,1610612752,NYK,New York Knicks,22501190,2026-04-12,NYK vs. CHA,L,240,96,...,12,27,39,25,10,3,10,25,-14.0,True
2,22025,1610612737,ATL,Atlanta Hawks,22501189,2026-04-12,ATL @ MIA,L,241,117,...,13,33,46,31,3,2,16,17,-26.0,False
3,22025,1610612762,UTA,Utah Jazz,22501198,2026-04-12,UTA @ LAL,L,240,107,...,17,25,42,25,13,4,16,14,-24.0,False
4,22025,1610612743,DEN,Denver Nuggets,22501197,2026-04-12,DEN @ SAS,W,239,128,...,19,39,58,30,4,5,10,16,10.0,False


In [ ]:
# 5: build a table with one row per game, with columns GAME_DATE, home_team, away_team, home_pts, away_pts
import numpy as np

one_row = matchups.copy()

# regex: look for "vs." in the MATCHUP column, and count how many times it occurs per GAME_ID. 
# If it is 0, then it's a neutral site game.
old_home_count = one_row.groupby('GAME_ID')['MATCHUP'].transform(
    lambda m: m.str.contains('vs.', regex=False).sum())
neutral_ids = one_row.loc[old_home_count == 0, 'GAME_ID'].unique()

# parts: vector separating abbreviations and vs/@
parts = one_row['MATCHUP'].str.split(' ')

# listed as home if the second part is "vs.", otherwise listed as away
listed_home = np.where(parts.str[1] == 'vs.', parts.str[0], parts.str[2])

# creates the is_home column
# True if the TEAM_ABBREVIATION matches the listed home team
one_row['is_home'] = one_row['TEAM_ABBREVIATION'] == listed_home

# Split into home and away rows, keep only the columns we need, rename
home = one_row[one_row['is_home']][['GAME_ID', 'GAME_DATE', 'TEAM_ABBREVIATION', 'PTS']]
away = one_row[~one_row['is_home']][['GAME_ID', 'TEAM_ABBREVIATION', 'PTS']]

home = home.rename(columns={'PTS': 'home_pts', 'TEAM_ABBREVIATION': 'home_team'})
away = away.rename(columns={'PTS': 'away_pts', 'TEAM_ABBREVIATION': 'away_team'})

# One row per game; validate errors out if any game isn't exactly 1 home + 1 away
games_clean = home.merge(away, on='GAME_ID', how='inner', validate='one_to_one')
games_clean['GAME_DATE'] = pd.to_datetime(games_clean['GAME_DATE'])
games_clean['neutral_site'] = games_clean['GAME_ID'].isin(neutral_ids)

games_clean.to_csv("../data/games_clean_2025_26.csv", index=False)
games_clean.head()

Neutral-site games: 5
Home-court edge (pts): 1.726530612244898


,GAME_ID,GAME_DATE,home_team,home_pts,away_team,away_pts,neutral_site
0,22501192,2026-04-12,TOR,136,BKN,101,False
1,22501190,2026-04-12,NYK,96,CHA,110,False
2,22501189,2026-04-12,MIA,143,ATL,117,False
3,22501193,2026-04-12,DAL,149,CHI,128,False
4,22501191,2026-04-12,PHI,126,MIL,106,False
